# Experiment 3 — Fusing clinical metadata with CNN image predictions (PAD-UFES-20)

> **AI-generated code disclosure:** The code in this notebook was generated with Claude (Anthropic) and run, checked and adapted by the author.
> **Prompt used:** *[paste your full prompt here]*

**Research question:** does combining clinical metadata with image information classify the six lesion types better
than either modality alone?

**Design (everything on the same team split — 1,838 train / 460 test images, grouped by patient):**

| Step | What happens |
|---|---|
| Inputs | `metadata.csv` (21 clinical features, same selection as `TabularDataAnalysis.ipynb`, `biopsed` excluded) and `cnn_probs_oof.csv` (EfficientNet-B0 class probabilities: **out-of-fold** for training images, final model for test images) |
| Candidates | metadata-only (LR, RF, gradient boosting, MLP), image-only (CNN), and fusion: late fusion (weighted average / log-linear), stacking (LR, gradient boosting) and **MLP fusion** |
| Selection | **5-fold patient-grouped cross-validation** on the training set (the team folds), macro-F1 |
| Final test | each candidate refitted on all training images, the 460 test images used **once**; patient-level bootstrap 95 % confidence intervals |

Why out-of-fold image probabilities: a stacking model must see image predictions that behave like predictions on
**unseen** images. Using the final CNN's predictions on its own training images would make the image input look
far more reliable than it is, and the fusion model would over-trust it.

## 0. Setup

In [ ]:
import warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, classification_report, confusion_matrix)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ---------- locate input files (Kaggle: /kaggle/input/... ; locally: this folder) ----------
SEARCH_DIRS = [Path("/kaggle/input"), Path(".")]
def find_file(name):
    for d in SEARCH_DIRS:
        if d.exists():
            hit = next(d.rglob(name), None)
            if hit is not None:
                return hit
    raise FileNotFoundError(f"{name} not found - attach the dataset / upload the file")

META_CSV  = find_file("metadata.csv")
PROBS_CSV = find_file("cnn_probs_oof.csv")
OUT_DIR   = Path("fusion_outputs"); OUT_DIR.mkdir(exist_ok=True)
print("metadata:", META_CSV, "\nimage probabilities:", PROBS_CSV)

SEED   = 42
N_BOOT = 2000
CLASSES   = ["NEV", "ACK", "SEK", "BCC", "SCC", "MEL"]          # same order as the CNN notebooks
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}
MAL_IDX   = [LABEL_MAP[c] for c in ["BCC", "SCC", "MEL"]]
P_COLS    = [f"p_{c}" for c in CLASSES]

# chart colours (validated categorical palette: metadata / image / fusion)
COL = {"Metadata": "#2a78d6", "Image": "#eb6834", "Fusion": "#1baf7a"}
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

## 1. Load and join metadata with the image predictions
Every image row gets the clinical record of its lesion (images of the same lesion share one record).

In [ ]:
meta  = pd.read_csv(META_CSV)
probs = pd.read_csv(PROBS_CSV)
df = probs.merge(meta, on="img_id", how="left", suffixes=("", "_meta"), validate="one_to_one")
assert df["diagnostic_meta"].notna().all(), "some images have no metadata row"
assert (df["diagnostic_meta"] == df["diagnostic"]).all() and (df["patient_id_meta"] == df["patient_id"]).all()
df["y"] = df["diagnostic"].map(LABEL_MAP)

dev  = df[(df.split == "train") & (df.source == "out_of_fold")].reset_index(drop=True)
test = df[df.split == "test"].reset_index(drop=True)
assert not set(dev.patient_id) & set(test.patient_id), "patient leakage!"
print(f"development (train) images: {len(dev)} | test images: {len(test)} | "
      f"excluded duplicate images: {(df.source == 'no_oof_prediction').sum()}")
print("CV folds:", dev.fold.value_counts().sort_index().to_dict())
pd.DataFrame({"train": dev.diagnostic.value_counts(), "test": test.diagnostic.value_counts()}).loc[CLASSES]

## 2. Clinical features — same selection and preprocessing as `TabularDataAnalysis.ipynb`
* 21 features; identifiers, the target and **`biopsed` excluded** (BCC, SCC and MEL are always biopsied → leaks the label).
* Numerical (`age`, `diameter_1`, `diameter_2`) and ordinal (`fitspatrick`): median imputation (+ scaling for LR / MLP).
* Categorical: missing → `"Missing"` category, then one-hot encoding.
* All preprocessing is **fitted inside each training fold only** (scikit-learn pipelines), so nothing leaks from validation or test rows.

**Structured missingness (worth one sentence in the report):** the 13 questionnaire fields are missing *only* for
ACK, NEV and SEK lesions — never for a cancer. Missingness itself is therefore informative; the models can exploit it
through the `"Missing"` category. This is a property of how PAD-UFES-20 was collected, not leakage we introduced,
but it partly explains why the metadata is strong at separating cancer from non-cancer.

In [ ]:
NUM = ["age", "diameter_1", "diameter_2"]
ORD = ["fitspatrick"]
CAT = ["smoke", "drink", "background_father", "background_mother", "pesticide", "gender", "skin_cancer_history",
       "cancer_history", "has_piped_water", "has_sewage_system", "region", "itch", "grew", "hurt", "changed",
       "bleed", "elevation"]
META_FEATURES = NUM + ORD + CAT
assert len(META_FEATURES) == 21 and "biopsed" not in META_FEATURES

q_missing = df["gender"].isna()
print("Questionnaire missing vs diagnosis (all images):")
print(pd.crosstab(np.where(q_missing, "questionnaire missing", "questionnaire present"), df.diagnostic)[CLASSES])

def clean_categories(X):
    X = pd.DataFrame(X).astype(object)
    return X.where(X.notna(), "Missing").astype(str)

def safe_log(X):
    return np.log(np.clip(np.asarray(X, dtype=float), 1e-6, 1.0))

def make_preprocessor(use_meta=True, use_img=False, scale=True):
    parts = []
    if use_meta:
        num_steps = [("impute", SimpleImputer(strategy="median"))] + ([("scale", StandardScaler())] if scale else [])
        parts.append(("num", Pipeline(num_steps), NUM + ORD))
        parts.append(("cat", Pipeline([("clean", FunctionTransformer(clean_categories)),
                                       ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT))
    if use_img:   # image probabilities enter as log-probabilities
        img_steps = [("log", FunctionTransformer(safe_log))] + ([("scale", StandardScaler())] if scale else [])
        parts.append(("img", Pipeline(img_steps), P_COLS))
    return ColumnTransformer(parts)

## 3. Candidate models
Class imbalance is handled the same way everywhere: `class_weight="balanced"` (or balanced sample weights for the MLPs),
so MEL and SCC are not ignored. The MLPs are soft-voting ensembles of 5 networks with different random seeds,
which makes their results much more stable on a dataset this small.

In [ ]:
def mlp_ensemble(hidden, alpha):
    return VotingClassifier([(f"seed{s}", MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha, activation="relu",
                                                        learning_rate_init=1e-3, max_iter=600, random_state=s))
                             for s in range(5)], voting="soft")

CANDIDATES = {
    # ----- metadata only -----
    "Meta: Logistic Regression": dict(group="Metadata", meta=True, img=False, scale=True,
        est=LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000)),
    "Meta: Random Forest": dict(group="Metadata", meta=True, img=False, scale=False,
        est=RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=SEED)),
    "Meta: Gradient Boosting": dict(group="Metadata", meta=True, img=False, scale=False,
        est=HistGradientBoostingClassifier(learning_rate=0.05, max_iter=250, max_leaf_nodes=15,
                                           l2_regularization=1.0, class_weight="balanced", random_state=SEED)),
    "Meta: MLP": dict(group="Metadata", meta=True, img=False, scale=True, sample_weight=True,
        est=mlp_ensemble((64,), 1e-2)),
    # ----- fusion: stacking (metadata + image log-probabilities as features) -----
    "Fusion: Logistic Regression": dict(group="Fusion", meta=True, img=True, scale=True,
        est=LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000)),
    "Fusion: Gradient Boosting": dict(group="Fusion", meta=True, img=True, scale=False,
        est=HistGradientBoostingClassifier(learning_rate=0.05, max_iter=250, max_leaf_nodes=15,
                                           l2_regularization=1.0, class_weight="balanced", random_state=SEED)),
    "Fusion: MLP (64)": dict(group="Fusion", meta=True, img=True, scale=True, sample_weight=True,
        est=mlp_ensemble((64,), 1e-2)),
    "Fusion: MLP (128-64)": dict(group="Fusion", meta=True, img=True, scale=True, sample_weight=True,
        est=mlp_ensemble((128, 64), 1e-2)),
}

def balanced_oversample(y, rng):
    counts = np.bincount(y, minlength=len(CLASSES)); target = counts.max()
    return np.concatenate([rng.choice(np.where(y == c)[0], target, replace=True) for c in range(len(CLASSES)) if counts[c]])

def fit_candidate(spec, X, y):
    pipe = Pipeline([("prep", make_preprocessor(spec["meta"], spec["img"], spec["scale"])), ("clf", clone(spec["est"]))])
    if spec.get("sample_weight"):
        try:
            pipe.fit(X, y, clf__sample_weight=compute_sample_weight("balanced", y))
        except TypeError:   # older scikit-learn: MLP without sample_weight -> balanced oversampling instead
            idx = balanced_oversample(y, np.random.default_rng(SEED))
            pipe.fit(X.iloc[idx], y[idx])
    else:
        pipe.fit(X, y)
    return pipe

def proba(model, X):
    """Class probabilities in CLASSES order, whatever classes the model saw."""
    p = np.zeros((len(X), len(CLASSES)))
    p[:, model.classes_] = model.predict_proba(X)
    return p

## 4. Five-fold patient-grouped cross-validation (training set only)
Same folds as the CNN cross-validation. Every candidate produces an out-of-fold prediction for every training image.

In [ ]:
def macro_f1(y, p):
    return f1_score(y, p.argmax(1), average="macro", labels=range(len(CLASSES)), zero_division=0)

folds = sorted(dev.fold.unique())
X_dev, y_dev = dev, dev["y"].to_numpy()
oof = {"Image: EfficientNet-B0": dev[P_COLS].to_numpy()}
fold_scores = {"Image: EfficientNet-B0": [macro_f1(y_dev[dev.fold == k], oof["Image: EfficientNet-B0"][dev.fold == k]) for k in folds]}
groups = {"Image: EfficientNet-B0": "Image"}

t0 = time.time()
for name, spec in CANDIDATES.items():
    p_oof = np.zeros((len(dev), len(CLASSES)))
    for k in folds:
        tr, va = (dev.fold != k).to_numpy(), (dev.fold == k).to_numpy()
        model = fit_candidate(spec, X_dev[tr], y_dev[tr])
        p_oof[va] = proba(model, X_dev[va])
    oof[name], groups[name] = p_oof, spec["group"]
    fold_scores[name] = [macro_f1(y_dev[dev.fold == k], p_oof[dev.fold == k]) for k in folds]
    print(f"{name:32s} CV macro-F1 {np.mean(fold_scores[name]):.3f} ± {np.std(fold_scores[name], ddof=1):.3f}"
          f"   ({time.time()-t0:.0f}s)")

### Late fusion (weighted average of the image and metadata predictions)
The metadata model used here is the best metadata-only candidate from the CV above. Two combination rules:
* **weighted average:** p = w · p_image + (1 − w) · p_metadata
* **log-linear (product of experts):** log p ∝ w · log p_image + (1 − w) · log p_metadata

The weight *w* is chosen **inside** the cross-validation (on the other four folds each time), so the reported CV score
is not optimistic.

In [ ]:
meta_names = [n for n in CANDIDATES if CANDIDATES[n]["group"] == "Metadata"]
BEST_META = max(meta_names, key=lambda n: np.mean(fold_scores[n]))
print("Metadata model used for late fusion:", BEST_META)
W_GRID = np.round(np.linspace(0, 1, 21), 2)

def combine(p_img, p_meta, w, rule):
    if rule == "average":
        return w * p_img + (1 - w) * p_meta
    logp = w * safe_log(p_img) + (1 - w) * safe_log(p_meta)
    e = np.exp(logp - logp.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

LATE = {"Fusion: Late (weighted average)": "average", "Fusion: Late (log-linear)": "loglinear"}
late_curves, late_w = {}, {}
for name, rule in LATE.items():
    p_img, p_meta = oof["Image: EfficientNet-B0"], oof[BEST_META]
    p_out, scores = np.zeros_like(p_img), []
    for k in folds:
        va = (dev.fold == k).to_numpy(); tr = ~va
        w_k = max(W_GRID, key=lambda w: macro_f1(y_dev[tr], combine(p_img[tr], p_meta[tr], w, rule)))
        p_out[va] = combine(p_img[va], p_meta[va], w_k, rule)
        scores.append(macro_f1(y_dev[va], p_out[va]))
    oof[name], fold_scores[name], groups[name] = p_out, scores, "Fusion"
    late_curves[name] = [macro_f1(y_dev, combine(p_img, p_meta, w, rule)) for w in W_GRID]
    late_w[name] = W_GRID[int(np.argmax(late_curves[name]))]
    print(f"{name:32s} CV macro-F1 {np.mean(scores):.3f} ± {np.std(scores, ddof=1):.3f}   (final w = {late_w[name]})")

In [ ]:
cv_table = pd.DataFrame({
    "group": groups,
    "cv_macro_f1": {n: np.mean(s) for n, s in fold_scores.items()},
    "cv_std": {n: np.std(s, ddof=1) for n, s in fold_scores.items()},
    "oof_balanced_acc": {n: balanced_accuracy_score(y_dev, p.argmax(1)) for n, p in oof.items()},
    "oof_accuracy": {n: accuracy_score(y_dev, p.argmax(1)) for n, p in oof.items()},
}).sort_values("cv_macro_f1", ascending=False).round(4)
cv_table.to_csv(OUT_DIR / "fusion_cv_results.csv")

BEST_FUSION = cv_table[cv_table.group == "Fusion"].index[0]
print("Selected by CV ->  best metadata:", BEST_META, "| best fusion:", BEST_FUSION)
cv_table

In [ ]:
order = cv_table.index[::-1]
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(order) + 1))
ax.barh(range(len(order)), cv_table.loc[order, "cv_macro_f1"], height=0.62,
        color=[COL[cv_table.loc[n, "group"]] for n in order], edgecolor="white", linewidth=2)
ax.errorbar(cv_table.loc[order, "cv_macro_f1"], range(len(order)), xerr=cv_table.loc[order, "cv_std"],
            fmt="none", ecolor=INK, elinewidth=1, capsize=3)
for i, n in enumerate(order):
    ax.text(cv_table.loc[n, "cv_macro_f1"] + cv_table.loc[n, "cv_std"] + 0.008, i,
            f"{cv_table.loc[n, 'cv_macro_f1']:.3f}", va="center", color=INK, fontsize=9)
ax.set_yticks(range(len(order)), order); ax.set_xlim(0, 1); ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_xlabel("5-fold CV macro-F1 (mean ± std)")
ax.set_title("Cross-validation: metadata vs image vs fusion", loc="left", color=INK)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in COL.values()], labels=list(COL), frameon=False,
          loc="lower right")
plt.tight_layout(); plt.savefig(OUT_DIR / "fusion_cv_comparison.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for (name, curve), ls in zip(late_curves.items(), ["-", "--"]):
    ax.plot(W_GRID, curve, ls, color=COL["Fusion"], lw=2, label=name.replace("Fusion: Late ", ""))
ax.set_xlabel("image weight w   (0 = metadata only, 1 = image only)"); ax.set_ylabel("OOF macro-F1")
ax.grid(color=GRID, lw=0.8); ax.legend(frameon=False); ax.set_title("Late fusion: effect of the image weight", loc="left", color=INK)
plt.tight_layout(); plt.savefig(OUT_DIR / "late_fusion_weight.png", dpi=200); plt.show()

## 5. Final models and the single test evaluation
Each candidate is refitted on **all** training images and evaluated once on the 460 test images. The model *choice*
(best metadata model, best fusion model) was already made by cross-validation above, so the test set only reports.

In [ ]:
X_test, y_test = test, test["y"].to_numpy()
test_probs = {"Image: EfficientNet-B0": test[P_COLS].to_numpy()}
final_models = {}
for name, spec in CANDIDATES.items():
    final_models[name] = fit_candidate(spec, X_dev, y_dev)
    test_probs[name] = proba(final_models[name], X_test)
for name, rule in LATE.items():
    test_probs[name] = combine(test_probs["Image: EfficientNet-B0"], test_probs[BEST_META], late_w[name], rule)

def evaluate(p, y):
    pred = p.argmax(1); p_cancer = p[:, MAL_IDX].sum(1)
    yb, pb = np.isin(y, MAL_IDX).astype(int), (p_cancer >= 0.5).astype(int)
    rec = recall_score(y, pred, average=None, labels=range(len(CLASSES)), zero_division=0)
    return dict(accuracy=accuracy_score(y, pred), balanced_accuracy=balanced_accuracy_score(y, pred),
                macro_precision=precision_score(y, pred, average="macro", zero_division=0),
                macro_recall=recall_score(y, pred, average="macro", zero_division=0),
                macro_f1=f1_score(y, pred, average="macro", zero_division=0),
                weighted_f1=f1_score(y, pred, average="weighted", zero_division=0),
                MEL_recall=rec[LABEL_MAP["MEL"]], SCC_recall=rec[LABEL_MAP["SCC"]],
                binary_accuracy=accuracy_score(yb, pb), binary_sensitivity=recall_score(yb, pb),
                binary_specificity=recall_score(yb, pb, pos_label=0), binary_roc_auc=roc_auc_score(yb, p_cancer))

test_table = pd.DataFrame({n: evaluate(p, y_test) for n, p in test_probs.items()}).T
test_table.insert(0, "group", pd.Series(groups))
test_table.insert(1, "cv_macro_f1", cv_table["cv_macro_f1"])
test_table = test_table.loc[cv_table.index].round(4)
test_table.to_csv(OUT_DIR / "fusion_test_results.csv")
test_table

### Headline comparison with patient-level bootstrap confidence intervals
The test set has only 13 MEL and 37 SCC images, so single numbers can move by a few points by chance. Resampling
**patients** (not images) 2,000 times gives 95 % intervals and a paired test of the fusion gain.

In [ ]:
HEADLINE = {"Metadata only": BEST_META, "Image only": "Image: EfficientNet-B0", "Fusion": BEST_FUSION}
rng = np.random.default_rng(SEED)
pat = test["patient_id"].to_numpy(); uniq = np.unique(pat)
rows_of = {p: np.where(pat == p)[0] for p in uniq}
boot = {k: [] for k in HEADLINE}
for _ in range(N_BOOT):
    idx = np.concatenate([rows_of[p] for p in rng.choice(uniq, len(uniq), replace=True)])
    for label, name in HEADLINE.items():
        boot[label].append(f1_score(y_test[idx], test_probs[name][idx].argmax(1), average="macro",
                                    labels=range(len(CLASSES)), zero_division=0))
boot = pd.DataFrame(boot); boot.to_csv(OUT_DIR / "bootstrap_macro_f1.csv", index=False)

summary = pd.DataFrame({
    "model": list(HEADLINE.values()),
    "test_macro_f1": [test_table.loc[n, "macro_f1"] for n in HEADLINE.values()],
    "ci_low": boot.quantile(0.025).values, "ci_high": boot.quantile(0.975).values,
    "test_balanced_acc": [test_table.loc[n, "balanced_accuracy"] for n in HEADLINE.values()],
    "test_accuracy": [test_table.loc[n, "accuracy"] for n in HEADLINE.values()],
    "binary_roc_auc": [test_table.loc[n, "binary_roc_auc"] for n in HEADLINE.values()],
}, index=list(HEADLINE)).round(3)
summary.to_csv(OUT_DIR / "headline_comparison.csv")

for other in ["Image only", "Metadata only"]:
    d = boot["Fusion"] - boot[other]
    print(f"Fusion − {other}: {d.mean():+.3f} macro-F1  (95% CI {d.quantile(.025):+.3f} to {d.quantile(.975):+.3f}; "
          f"fusion better in {(d > 0).mean():.1%} of resamples)")
summary

In [ ]:
labels = list(HEADLINE); colors = [COL["Metadata"], COL["Image"], COL["Fusion"]]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
x = np.arange(len(labels))
ax.bar(x, summary["test_macro_f1"], width=0.55, color=colors, edgecolor="white", linewidth=2)
ax.errorbar(x, summary["test_macro_f1"], yerr=[summary["test_macro_f1"] - summary["ci_low"],
            summary["ci_high"] - summary["test_macro_f1"]], fmt="none", ecolor=INK, elinewidth=1, capsize=4)
for i, v in enumerate(summary["test_macro_f1"]):
    ax.text(i, summary["ci_high"].iloc[i] + 0.015, f"{v:.3f}", ha="center", color=INK)
ax.set_xticks(x, [f"{l}\n({HEADLINE[l].split(': ')[-1]})" for l in labels], fontsize=9)
ax.set_ylim(0, 1); ax.set_ylabel("Test macro-F1 (95% bootstrap CI)"); ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_title("Same 460 test images: metadata vs image vs fusion", loc="left", color=INK)
plt.tight_layout(); plt.savefig(OUT_DIR / "headline_test_comparison.png", dpi=200); plt.show()

### Where does fusion help? Per-class F1 and confusion matrices

In [ ]:
per_class = pd.DataFrame({label: f1_score(y_test, test_probs[n].argmax(1), average=None, labels=range(len(CLASSES)),
                                          zero_division=0) for label, n in HEADLINE.items()}, index=CLASSES).round(3)
per_class["n_test"] = pd.Series(y_test).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES).values
per_class.to_csv(OUT_DIR / "per_class_f1.csv")

fig, ax = plt.subplots(figsize=(8.5, 3.8)); w = 0.26
for j, (label, c) in enumerate(zip(HEADLINE, colors)):
    ax.bar(np.arange(len(CLASSES)) + (j - 1) * w, per_class[label], width=w - 0.02, color=c, label=label,
           edgecolor="white", linewidth=1)
ax.set_xticks(range(len(CLASSES)), [f"{c}\n(n={n})" for c, n in zip(CLASSES, per_class.n_test)])
ax.set_ylim(0, 1); ax.set_ylabel("Test F1"); ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=3, loc="upper left"); ax.set_title("Per-class F1 on the test set", loc="left", color=INK)
plt.tight_layout(); plt.savefig(OUT_DIR / "per_class_f1.png", dpi=200); plt.show()

cmap = LinearSegmentedColormap.from_list("blue", ["#fcfcfb", "#86b6ef", "#2a78d6", "#184f95"])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (label, name) in zip(axes, [("Image only", HEADLINE["Image only"]), ("Fusion", BEST_FUSION)]):
    cm = confusion_matrix(y_test, test_probs[name].argmax(1), labels=range(len(CLASSES)))
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap=cmap, vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.6 else INK)
    ax.set_xticks(range(len(CLASSES)), CLASSES); ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{label}: {name}", loc="left", color=INK, fontsize=10)
plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_image_vs_fusion.png", dpi=200); plt.show()
per_class

In [ ]:
print(f"===== {BEST_FUSION} (selected by CV) — full test report =====")
print(classification_report(y_test, test_probs[BEST_FUSION].argmax(1), labels=range(len(CLASSES)),
                            target_names=CLASSES, digits=3, zero_division=0))
pd.concat([test[["img_id", "patient_id", "diagnostic"]],
           pd.DataFrame(test_probs[BEST_FUSION], columns=P_COLS)], axis=1).assign(
    predicted=[CLASSES[i] for i in test_probs[BEST_FUSION].argmax(1)]).to_csv(OUT_DIR / "fusion_test_predictions.csv", index=False)
print("All outputs saved to", OUT_DIR.resolve())

## 6. Notes for the report
* **All three experiments share one split** (1,838 / 460 images, grouped by patient). The metadata-only numbers here
  therefore differ from `TabularDataAnalysis.ipynb`, which used one row per lesion and its own split (381 test lesions)
  and no class weighting.
* **Model selection used cross-validation only**; the test set was evaluated once per model and the "best" models were
  fixed before looking at it.
* **Stacking uses out-of-fold image probabilities** for training; test images use the final CNN. The final CNN was trained
  on slightly more data than each fold model, so its test probabilities can be a little more confident — a known,
  small mismatch in stacking.
* **Structured missingness:** questionnaire fields are only missing for non-cancer lesions; models can use this signal.
* Evaluation is per image (as in the CNN notebooks); images of the same lesion share one clinical record.